# Draft 01: Fast sklearn Baseline

Small, fast regression baseline for `cpe-232-insurance-premium-prediction` that works in both local and Kaggle runtimes.


In [ ]:
import importlib.util
import subprocess
import sys


def run_cmd(cmd):
    print("Running:", " ".join(cmd))
    subprocess.check_call(cmd)


def ensure_package(module_name, package_name=None):
    pkg = package_name or module_name
    if importlib.util.find_spec(module_name) is not None:
        print(f"{pkg} is already installed")
        return
    run_cmd([sys.executable, "-m", "pip", "install", pkg])


ensure_package("numpy")
ensure_package("pandas")
ensure_package("dotenv", "python-dotenv")
ensure_package("kaggle")
ensure_package("sklearn", "scikit-learn")


In [ ]:
import os
from datetime import UTC, datetime
from pathlib import Path
from zipfile import ZipFile

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from IPython.display import display
from kaggle.api.kaggle_api_extended import KaggleApi
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

pd.set_option("display.max_columns", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.6f}")


In [ ]:
COMPETITION = "cpe-232-insurance-premium-prediction"
RUN_DOWNLOAD = True
RUN_SUBMISSION = True
FORCE_DOWNLOAD = True
RANDOM_STATE = 42
FAST_SAMPLE_ROWS = 100_000
VALID_SIZE = 0.2

ID_COL = "id"
LABEL_COL = "Premium Amount"
NOTEBOOK_SLUG = "ins232-01-fast-sklearn-baseline"


def load_env_from_candidates():
    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd.parent, cwd.parent.parent]
    for directory in candidates:
        env_path = directory / ".env"
        if env_path.exists():
            load_dotenv(env_path, override=True)
            print("Loaded env from", env_path)
            return env_path
    print("No .env file found in notebook parents, using current environment")
    return None


def resolve_data_paths_fallback():
    cwd = Path.cwd().resolve()
    kaggle_candidates = [
        Path(f"/kaggle/input/{COMPETITION}"),
        Path("/kaggle/input"),
    ]
    local_candidates = [
        cwd,
        cwd / "data",
        cwd.parent,
        cwd.parent / "data",
    ]

    candidate_triplets = []
    for base in kaggle_candidates + local_candidates:
        candidate_triplets.extend(
            [
                (base / "train.csv", base / "test.csv", base / "sample_submission.csv"),
                (
                    base / COMPETITION / "train.csv",
                    base / COMPETITION / "test.csv",
                    base / COMPETITION / "sample_submission.csv",
                ),
                (
                    base / "data" / "train.csv",
                    base / "data" / "test.csv",
                    base / "data" / "sample_submission.csv",
                ),
            ]
        )

    seen = set()
    for train_path, test_path, sample_path in candidate_triplets:
        key = (str(train_path), str(test_path), str(sample_path))
        if key in seen:
            continue
        seen.add(key)
        if train_path.exists() and test_path.exists() and sample_path.exists():
            return train_path, test_path, sample_path

    raise FileNotFoundError(
        "Could not find train/test/sample_submission files in Kaggle input or local folders"
    )


def prepare_competition_data(api_client, competition, data_dir, force_download=False):
    data_dir = Path(data_dir)
    data_dir.mkdir(parents=True, exist_ok=True)
    zip_path = data_dir / f"{competition}.zip"
    extract_dir = data_dir / competition

    if force_download or not zip_path.exists():
        files_resp = api_client.competition_list_files(competition)
        try:
            api_client.competition_download_files(
                competition=competition,
                path=str(data_dir),
                force=force_download,
                quiet=False,
            )
            print("Competition zip download complete")
        except Exception as download_error:
            print("Bulk download failed, fallback to per-file download:", download_error)
            for file_obj in files_resp.files:
                api_client.competition_download_file(
                    competition=competition,
                    file_name=file_obj.name,
                    path=str(data_dir),
                    force=force_download,
                    quiet=False,
                )

    if zip_path.exists():
        extract_dir.mkdir(parents=True, exist_ok=True)
        with ZipFile(zip_path) as zf:
            zf.extractall(extract_dir)
        print("Extracted zip to", extract_dir)

    train_path = extract_dir / "train.csv"
    test_path = extract_dir / "test.csv"
    sample_path = extract_dir / "sample_submission.csv"
    if not (train_path.exists() and test_path.exists() and sample_path.exists()):
        raise FileNotFoundError(
            f"Downloaded data for {competition} but expected CSV files were not found"
        )

    return train_path, test_path, sample_path, zip_path, extract_dir


def maybe_sample_training_rows(df, n_rows, random_state=RANDOM_STATE):
    if n_rows is None or n_rows <= 0 or len(df) <= n_rows:
        return df.copy(), False
    sampled_df = df.sample(n=n_rows, random_state=random_state).sort_index().reset_index(drop=True)
    return sampled_df, True


In [ ]:
load_env_from_candidates()
os.environ.pop("KAGGLE_API_TOKEN", None)

api = None
try:
    api = KaggleApi()
    api.authenticate()
    print("Authenticated user:", api.get_config_value("username"))
except Exception as auth_error:
    print("Kaggle API auth not ready:", auth_error)
    print("Will use existing /kaggle/input or local data if available")

if RUN_DOWNLOAD and api is not None:
    BASE_DIR = Path.cwd()
    DATA_DIR = BASE_DIR / "data"
    TRAIN_PATH, TEST_PATH, SAMPLE_PATH, ZIP_PATH, EXTRACT_DIR = prepare_competition_data(
        api_client=api,
        competition=COMPETITION,
        data_dir=DATA_DIR,
        force_download=FORCE_DOWNLOAD,
    )
else:
    TRAIN_PATH, TEST_PATH, SAMPLE_PATH = resolve_data_paths_fallback()
    ZIP_PATH = None
    EXTRACT_DIR = TRAIN_PATH.parent
    BASE_DIR = TRAIN_PATH.parent
    if RUN_DOWNLOAD and api is None:
        print("RUN_DOWNLOAD=True but auth is unavailable, using fallback paths instead")

OUTPUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
OUTPUT_PATH = OUTPUT_DIR / f"submission_{NOTEBOOK_SLUG}.csv"
DIAGNOSTICS_PATH = OUTPUT_DIR / f"diagnostics_{NOTEBOOK_SLUG}.csv"

config_df = pd.DataFrame(
    [
        {
            "competition": COMPETITION,
            "run_download": RUN_DOWNLOAD,
            "run_submission": RUN_SUBMISSION,
            "force_download": FORCE_DOWNLOAD,
            "random_state": RANDOM_STATE,
            "fast_sample_rows": FAST_SAMPLE_ROWS,
            "valid_size": VALID_SIZE,
            "train_path": str(TRAIN_PATH),
            "test_path": str(TEST_PATH),
            "sample_path": str(SAMPLE_PATH),
            "output_file": str(OUTPUT_PATH),
            "diagnostics_file": str(DIAGNOSTICS_PATH),
        }
    ]
)
display(config_df)


In [ ]:
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)
sample_submission = pd.read_csv(SAMPLE_PATH)

train_model_df, used_sampling = maybe_sample_training_rows(train_df, FAST_SAMPLE_ROWS)

feature_cols = [c for c in train_model_df.columns if c not in [ID_COL, LABEL_COL]]
X = train_model_df[feature_cols].copy()
y = train_model_df[LABEL_COL].copy()
X_test = test_df[feature_cols].copy()

numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = [c for c in feature_cols if c not in numeric_cols]

dataset_summary = pd.DataFrame(
    [
        {
            "train_rows_full": len(train_df),
            "train_rows_used": len(train_model_df),
            "test_rows": len(test_df),
            "used_sampling": used_sampling,
            "numeric_feature_count": len(numeric_cols),
            "categorical_feature_count": len(categorical_cols),
            "target_mean": float(y.mean()),
            "target_std": float(y.std()),
            "target_min": float(y.min()),
            "target_max": float(y.max()),
        }
    ]
)
display(dataset_summary)
display(train_model_df.head(3))


In [ ]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=VALID_SIZE,
    random_state=RANDOM_STATE,
)

numeric_transformer = Pipeline(
    [
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_transformer = Pipeline(
    [
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=True)),
    ]
)

preprocessor = ColumnTransformer(
    [
        ("num", numeric_transformer, numeric_cols),
        ("cat", categorical_transformer, categorical_cols),
    ]
)

model = Pipeline(
    [
        ("preprocessor", preprocessor),
        ("regressor", Ridge(alpha=2.0)),
    ]
)

model.fit(X_train, y_train)
valid_predictions = np.clip(model.predict(X_valid), 0, None)
validation_mae = mean_absolute_error(y_valid, valid_predictions)

validation_preview = pd.DataFrame(
    {
        "actual": y_valid.reset_index(drop=True).head(10),
        "predicted": pd.Series(valid_predictions).head(10),
    }
)
display(validation_preview)
print("Validation MAE:", f"{validation_mae:,.6f}")


In [ ]:
final_model = Pipeline(
    [
        ("preprocessor", preprocessor),
        ("regressor", Ridge(alpha=2.0)),
    ]
)
final_model.fit(X, y)
test_predictions = np.clip(final_model.predict(X_test), 0, None)

submission_df = sample_submission.copy()
submission_df[LABEL_COL] = test_predictions

assert submission_df.columns.tolist() == [ID_COL, LABEL_COL], "Submission schema mismatch"
assert len(submission_df) == len(test_df), "Submission length mismatch"

diagnostics_df = pd.DataFrame(
    [
        {
            "competition": COMPETITION,
            "notebook_slug": NOTEBOOK_SLUG,
            "validation_mae": float(validation_mae),
            "train_rows_full": int(len(train_df)),
            "train_rows_used": int(len(train_model_df)),
            "used_sampling": bool(used_sampling),
            "fast_sample_rows": int(FAST_SAMPLE_ROWS),
            "numeric_feature_count": int(len(numeric_cols)),
            "categorical_feature_count": int(len(categorical_cols)),
            "target_mean": float(y.mean()),
            "target_std": float(y.std()),
            "target_min": float(y.min()),
            "target_max": float(y.max()),
            "test_prediction_mean": float(np.mean(test_predictions)),
            "test_prediction_std": float(np.std(test_predictions)),
            "test_prediction_min": float(np.min(test_predictions)),
            "test_prediction_max": float(np.max(test_predictions)),
        }
    ]
)

submission_df.to_csv(OUTPUT_PATH, index=False)
diagnostics_df.to_csv(DIAGNOSTICS_PATH, index=False)

display(diagnostics_df)
display(submission_df.head(10))
print("Saved submission to", OUTPUT_PATH)
print("Saved diagnostics to", DIAGNOSTICS_PATH)


In [ ]:
if RUN_SUBMISSION and api is not None:
    submit_message = (
        f"{NOTEBOOK_SLUG} ridge baseline | "
        f"val_mae={validation_mae:.6f} "
        f"time={datetime.now(UTC).strftime('%Y-%m-%d %H:%M:%S')}Z"
    )
    response = api.competition_submit(
        file_name=str(OUTPUT_PATH),
        message=submit_message,
        competition=COMPETITION,
        quiet=False,
    )
    print("Submission response:", response)
elif RUN_SUBMISSION:
    print("RUN_SUBMISSION is True but Kaggle auth is unavailable. File is ready at", OUTPUT_PATH)
else:
    print("RUN_SUBMISSION is False. File is ready at", OUTPUT_PATH)
